# Eight wells in 2D: Prinz × double well
[Open in Colab](https://colab.research.google.com/github/paulrobustelli/CSMP_150_2026/blob/main/notebooks/Prinz_2D_Eight_Well_MSM.ipynb)

**A compact teaching example:** four wells in x, two in y, and an eight-state MSM. This is our explicitly defined 2D extension, not a canonical published 2D Prinz potential.

$$V(x,y)=V_P(x)+V_D(y),$$
$$V_P(x)=2x^8+0.8e^{-80x^2}+0.2e^{-80(x-0.5)^2}+0.5e^{-40(x+0.5)^2},\qquad V_D(y)=0.6\left[(y/0.55)^2-1\right]^2.$$

We use reversible local Metropolis dynamics on a fixed 40×40 grid, with $\beta=5$. Time is measured in **MC steps**, not ns. The fine-grid chain generates trajectories; its 1,600 positions are grouped into **eight basin states** for the MSM.

**Story:** look at the landscape → sample it → estimate an 8×8 matrix → identify its modes → put all population in one basin → watch it relax. The code is complete and can be used as an instructor demonstration; short exercises at the end invite students to repeat the estimation and change the model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from scipy.sparse.csgraph import connected_components
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.colors import PowerNorm, SymLogNorm
from IPython.display import display, Video
from pathlib import Path
np.set_printoptions(precision=4,suppress=True)
OUTPUT=Path(".")
N=40;beta=5.;q=.25
x=np.linspace(-1,1,N);y=x.copy();X,Y=np.meshgrid(x,y)
def prinz(x):
    return 2*x**8+.8*np.exp(-80*x*x)+.2*np.exp(-80*(x-.5)**2)+.5*np.exp(-40*(x+.5)**2)
def double_well(y):return .6*((y/.55)**2-1)**2
energy=prinz(X)+double_well(Y)
def chain_1d(energy):
    P=np.zeros((N,N))
    for i in range(N):
        for j in [i-1,i+1]:
            if 0<=j<N:P[i,j]=q*min(1.,np.exp(-beta*(energy[j]-energy[i])))
        P[i,i]=1-P[i].sum()
    w=np.exp(-beta*(energy-energy.min()));w/=w.sum()
    np.testing.assert_allclose(w[:,None]*P,w[None,:]*P.T,atol=1e-14)
    return P,w
Px,wx=chain_1d(prinz(x));Py,wy=chain_1d(double_well(y))
W=np.outer(wy,wx);w=W.ravel()
# State order: lower row A−, B−, C−, D−; upper row A+, B+, C+, D+.
labels=["A−","B−","C−","D−","A+","B+","C+","D+"]
xb=np.digitize(x,[-.5,0,.5]);yb=(y>=0).astype(int)
assignment=(4*yb[:,None]+xb[None,:]).ravel()
H=np.eye(8)[assignment];pi_exact=w@H
centers_x=np.array([-.75,-.25,.25,.75]);centers_y=np.array([-.5,.5])


## 1. Eight wells and their equilibrium populations
$$\mu(x,y)\propto e^{-\beta V(x,y)},\qquad \pi_s=\sum_{z\in s}w_z.$$
$w_z$ is the equilibrium probability of a fine-grid position; $\pi_s$ is the total probability of a basin. The y potential is symmetric, so corresponding upper and lower basins have equal equilibrium populations.

White lines mark the MSM boundaries, at x = −0.5, 0, 0.5 and y = 0. Color bars distinguish energy, probability per fine-grid cell, and probability per basin.

In [ ]:
def basin_grid(ax,annotate=True):
    for edge in [-.5,0,.5]:ax.axvline(edge,color="white",lw=.8,ls="--")
    ax.axhline(0,color="white",lw=.8,ls="--")
    if annotate:
        for s in range(8):ax.text(centers_x[s%4],centers_y[s//4],labels[s],ha="center",va="center",color="white",weight="bold")
fig=plt.figure(figsize=(14,4.3),layout="constrained")
a=fig.add_subplot(131,projection="3d");a.plot_surface(X,Y,energy,cmap="viridis",linewidth=0)
a.set(xlabel="x",ylabel="y",zlabel="Potential",title="Prinz(x) + double well(y)")
b=fig.add_subplot(132);im=b.imshow(energy,origin="lower",extent=(-1,1,-1,1),cmap="viridis")
basin_grid(b);b.set(title="Eight basins",xlabel="x",ylabel="y");fig.colorbar(im,ax=b,label="Potential energy")
a=fig.add_subplot(133);im=a.imshow(W,origin="lower",extent=(-1,1,-1,1),cmap="magma")
basin_grid(a);a.set(title="Equilibrium population",xlabel="x",ylabel="y");fig.colorbar(im,ax=a,label="Probability per grid cell")
fig.savefig(OUTPUT/"Prinz_8well_landscape.png",dpi=180);plt.show()
print("Basin order:",labels)
print("Equilibrium basin probabilities:",pi_exact)
np.testing.assert_allclose(pi_exact[:4],pi_exact[4:],atol=1e-14)


## 2. Run local MC trajectories
At every step, choose x or y with probability 1/2 and apply one step of that coordinate's Metropolis chain. Rejected moves count as steps. The particle's 3D height is always the potential evaluated at its current position.

Training and validation use independent seeds. Training starts at equilibrium; this supports the simple reversible count estimator used below. The later nonequilibrium experiment starts from one basin and is a separate propagation task.

In [ ]:
def simulate(steps,seed,start=None):
    rng=np.random.default_rng(seed)
    state=int(rng.choice(N*N,p=w)) if start is None else int(start)
    iy,ix=divmod(state,N)
    path=np.empty(steps+1,dtype=np.int16);path[0]=state
    axes=rng.integers(0,2,size=steps);draws=rng.random(steps)
    lx=np.r_[0,np.diag(Px,-1)];rx=np.r_[np.diag(Px,1),0]
    ly=np.r_[0,np.diag(Py,-1)];ry=np.r_[np.diag(Py,1),0]
    for t in range(steps):
        r=draws[t]
        if axes[t]==0:ix+=(-1 if r<lx[ix] else (1 if r<lx[ix]+rx[ix] else 0))
        else:iy+=(-1 if r<ly[iy] else (1 if r<ly[iy]+ry[iy] else 0))
        path[t+1]=iy*N+ix
    return path
STEPS_PER_PATH=250_000
training=[simulate(STEPS_PER_PATH,s) for s in range(7100,7108)]
validation=[simulate(STEPS_PER_PATH,s) for s in range(7200,7204)]
for path in training+validation:
    iy,ix=np.divmod(path.astype(int),N)
    assert np.all(np.abs(np.diff(ix))+np.abs(np.diff(iy))<=1)
observed=np.bincount(np.concatenate(training),minlength=N*N).astype(float);observed/=observed.sum()
fig,axes=plt.subplots(1,2,figsize=(10,4),layout="constrained")
xx=np.arange(8)
axes[0].bar(xx-.18,pi_exact,width=.36,label="Equilibrium")
axes[0].bar(xx+.18,observed@H,width=.36,label="Observed MC")
axes[0].set(xticks=xx,xticklabels=labels,ylabel="Basin probability",title="Sampling the eight basins");axes[0].legend()
im=axes[1].imshow(observed.reshape(N,N),origin="lower",extent=(-1,1,-1,1),cmap="magma")
basin_grid(axes[1]);axes[1].set(title="Observed fine-grid population",xlabel="x",ylabel="y")
fig.colorbar(im,ax=axes[1],label="Probability per grid cell")
fig.savefig(OUTPUT/"Prinz_8well_populations.png",dpi=180);plt.show()
print("Training steps:",len(training)*STEPS_PER_PATH,"Validation steps:",len(validation)*STEPS_PER_PATH)


## 3. Estimate the 8×8 MSM from counts
For lag $\tau$, $C_{ij}(\tau)$ counts transitions from basin i to basin j within each trajectory. Never count across trajectory boundaries.
$$F=(C+C^T)/2,\qquad T_{ij}=F_{ij}/\sum_jF_{ij},\qquad\widehat\pi_i=\sum_jF_{ij}/\sum_{ij}F_{ij}.$$
This is a reversible **symmetrized-count estimator**, not reversible maximum likelihood. The fitted equilibrium $\widehat\pi$ comes from counts; the known fine-grid equilibrium is a reference, not input to this fit.

The main demonstration uses lag 200 MC steps. Eight basins need not be Markovian at every lag; the checks below assess this approximation.

In [ ]:
def counts_at(paths,lag):
    C=np.zeros((8,8),dtype=np.int64)
    for path in paths:
        s=assignment[path]
        C+=np.bincount(s[:-lag]*8+s[lag:],minlength=64).reshape(8,8)
    return C
def fit(C):
    F=(C+C.T)/2;totals=F.sum(1)
    if np.any(totals==0) or connected_components(F>0,directed=False,return_labels=False)!=1:
        raise ValueError("Insufficient connected sampling: extend the simulation")
    return F/totals[:,None],totals/totals.sum()
LAG=200
C=counts_at(training,LAG);T,pi=fit(C)
np.testing.assert_allclose(T.sum(1),1,atol=1e-12)
np.testing.assert_allclose(pi@T,pi,atol=1e-12)
np.testing.assert_allclose(pi[:,None]*T,pi[None,:]*T.T,atol=1e-12)
fig,ax=plt.subplots(figsize=(7,6),layout="constrained")
im=ax.imshow(T,cmap="magma",vmin=0,vmax=1)
for i in range(8):
    for j in range(8):ax.text(j,i,f"{T[i,j]:.2f}",ha="center",va="center",fontsize=9,color="black" if T[i,j]>.6 else "white")
ax.set(xticks=range(8),yticks=range(8),xticklabels=labels,yticklabels=labels,xlabel="Destination basin",ylabel="Starting basin",title=f"Count-fitted MSM; lag {LAG} MC steps")
fig.colorbar(im,ax=ax,label="Transition probability")
fig.savefig(OUTPUT/"Prinz_8well_transition_matrix.png",dpi=180);plt.show()
print("Outgoing counts:",C.sum(1))


## 4. What are the dominant exchange patterns?
$$T\psi_i=\lambda_i\psi_i,\qquad\sum_s\widehat\pi_s\psi_i(s)\psi_j(s)=\delta_{ij}.$$
Each eigenvector has eight entries—one per basin. Plot them at their spatial locations. The first mode is constant. The other modes describe signed patterns of population imbalance. Their signs are arbitrary until combined with the initial amplitudes.

Below, diagonalize $D_{\widehat\pi}^{1/2}TD_{\widehat\pi}^{-1/2}$ to obtain real eigenvalues and equilibrium-normalized eigenvectors. The reported modes are properties of the **fitted eight-state model**.

In [ ]:
def eigensystem(T,pi):
    root=np.sqrt(pi);S=root[:,None]*T/root[None,:]
    np.testing.assert_allclose(S,S.T,atol=1e-12)
    vals,U=eigh(S);vals=vals[::-1];psi=U[:,::-1]/root[:,None]
    for i in range(8):
        if psi[np.argmax(abs(psi[:,i])),i]<0:psi[:,i]*=-1
    return vals,psi
vals,psi=eigensystem(T,pi)
np.testing.assert_allclose(T@psi,psi*vals,atol=1e-11)
np.testing.assert_allclose(psi.T@(pi[:,None]*psi),np.eye(8),atol=1e-11)
np.testing.assert_allclose(psi[:,0],1,atol=1e-10)
fig,axes=plt.subplots(2,3,figsize=(12,7),layout="constrained")
for i,ax in enumerate(axes.ravel()):
    field=psi[:,i].reshape(2,4);limit=max(abs(psi[:,i]))
    im=ax.imshow(field,origin="lower",extent=(-1,1,-1,1),aspect="equal",cmap="coolwarm",vmin=-limit,vmax=limit)
    for s in range(8):ax.text(centers_x[s%4],centers_y[s//4],f"{labels[s]}\n{psi[s,i]:.2f}",ha="center",va="center",fontsize=9)
    ax.set(title=f"Mode {i+1}; λ={vals[i]:.4f}",xlabel="x",ylabel="y");fig.colorbar(im,ax=ax)
fig.savefig(OUTPUT/"Prinz_8well_eigenmodes.png",dpi=180);plt.show()
print("Eigenvalues:",vals)
print("Timescales at this lag:",[-LAG/np.log(v) if 0<v<1 else np.nan for v in vals[1:]])


## 5. Refit at different lags; check predictions
$$t_i(\tau)=-\tau/\log\lambda_i(\tau),\qquad T(k\tau)\stackrel{?}{\approx}T(\tau)^k.$$
Refit from counts at each lag for the timescale plot. For the CK plot, compare powers of the lag-200 matrix with direct counts from held-out trajectories. Points are estimates, not exact truths. This compact lesson shows all eight return probabilities; the exercise extends the check to all 64 entries.

For a sampling-free comparison, exploit separability: the fine-grid random-axis chain has eigenvalues $(\lambda_a^x+\lambda_b^y)/2$ and product eigenvectors. This lets us calculate the exact basin-projected reference without fitting it to data.

In [ ]:
lags=np.array([25,50,100,200,500,1000]);its=[]
for lag in lags:
    fitted,mass=fit(counts_at(training,int(lag)));v,_=eigensystem(fitted,mass)
    its.append([-lag/np.log(z) if 0<z<1 else np.nan for z in v[1:5]])
fig,ax=plt.subplots(figsize=(8,4),layout="constrained")
for i in range(4):ax.plot(lags,np.array(its)[:,i],"o-",label=f"Mode {i+2}")
ax.set(xlabel="Lag (MC steps)",ylabel="Implied timescale (MC steps)",title="New count fits at each lag");ax.legend();plt.show()
ex,Ux=eigh(np.sqrt(wx)[:,None]*Px/np.sqrt(wx)[None,:])
ey,Uy=eigh(np.sqrt(wy)[:,None]*Py/np.sqrt(wy)[None,:])
reference_values=((ey[:,None]+ex[None,:])/2).ravel()
weighted_H=(np.sqrt(w)[:,None]*H).reshape(N,N,8)
Bproj=np.einsum("yb,xa,yxs->bas",Uy,Ux,weighted_H).reshape(N*N,8)
def reference(lag):return (Bproj.T@(reference_values[:,None]**lag*Bproj))/pi_exact[:,None]
np.testing.assert_allclose(reference(0),np.eye(8),atol=1e-12)
np.testing.assert_allclose(reference(LAG).sum(1),1,atol=1e-11)
K=np.arange(11)
predicted=np.array([np.linalg.matrix_power(T,int(k)) for k in K])
direct=[]
for k in K:
    if k==0:direct.append(np.eye(8))
    else:
        counts=counts_at(validation,int(k*LAG));direct.append(counts/counts.sum(1,keepdims=True))
direct=np.array(direct);exact=np.array([reference(int(k*LAG)) for k in K])
fig,axes=plt.subplots(2,4,figsize=(13,6),layout="constrained")
for s,ax in enumerate(axes.ravel()):
    ax.plot(K,predicted[:,s,s],label="MSM prediction")
    ax.plot(K,direct[:,s,s],"k.",label="Held-out counts")
    ax.plot(K,exact[:,s,s],"--",color="gray",label="Exact reference")
    ax.set(title=labels[s],xlabel="Lag steps k",ylim=(0,1),ylabel="Return probability")
axes[0,0].legend(fontsize=8)
fig.savefig(OUTPUT/"Prinz_8well_CK.png",dpi=180);plt.show()
print("Max all-entry error vs exact, k=1..10:",np.max(abs(predicted[1:]-exact[1:])))


## 6. Start with everything in A− and calculate the amplitudes
$$p_0=(1,0,0,0,0,0,0,0),\qquad u_{0,s}=p_{0,s}/\widehat\pi_s.$$
$$a_i=\sum_s p_{0,s}\psi_i(s)=\psi_i(A-).$$
$$p_k=p_0T^k,\qquad p_{k,s}=\widehat\pi_s\sum_{i=1}^8a_i\lambda_i^k\psi_i(s).$$
The stationary contribution is $a_1\psi_1=1$. All eight modes are needed to reconstruct the sharp initial condition exactly; at late times only the slowest excited modes remain. The limiting distribution is the **fitted** equilibrium $\widehat\pi$, which need not equal the reference exactly.

The heatmaps show probability **per basin**, not continuous density. An initial condition confined to an entire basin still requires a within-basin distribution to predict fine-grid dynamics; the MSM does not retain that information.

In [ ]:
p0=np.zeros(8);p0[0]=1
a=p0@psi
N_LAGS=200
population=np.empty((N_LAGS+1,8));population[0]=p0
for k in range(N_LAGS):population[k+1]=population[k]@T
spectral=np.array([pi*(psi@(a*vals**k)) for k in range(N_LAGS+1)])
np.testing.assert_allclose(spectral,population,atol=1e-10)
np.testing.assert_allclose(population.sum(1),1,atol=1e-12)
assert population.min()>-1e-12
TV=.5*abs(population-pi).sum(1)
print("Mode | initial amplitude | eigenvalue")
for i in range(8):print(f"{i+1:4d} | {a[i]:17.5f} | {vals[i]:.6f}")
print("Final distance to fitted equilibrium:",TV[-1])
fig,axes=plt.subplots(1,4,figsize=(14,3.6),layout="constrained")
for ax,k in zip(axes,[0,1,10,200]):
    ax.bar(np.arange(8),population[k],color="#0072b2")
    ax.plot(np.arange(8),pi,"k_",ms=12,label="Fitted equilibrium")
    ax.set(xticks=range(8),xticklabels=labels,ylim=(0,1),title=f"k={k}; {k*LAG:,} MC steps",ylabel="Probability")
axes[0].legend(fontsize=8)
fig.savefig(OUTPUT/"Prinz_8well_relaxation.png",dpi=180);plt.show()


## 7. Two movies, two different meanings
**MC trajectory:** one particle moves on the energy surface; the running histogram includes every visit. It is a time average and will fluctuate.

**Ensemble relaxation:** all initial probability is in A−. The movie updates eight populations, their excess/depletion relative to fitted equilibrium, and all seven nonstationary amplitudes. This is deterministic propagation of the fitted MSM, not a noisy single-particle histogram. Fixed nonlinear color scales reveal small probabilities without rescaling each frame.

The movie cell exports portable MP4 files and embeds them in Colab. Set EXPORT_MOVIES=False to skip rendering during quick experiments.

In [ ]:
import importlib.util,subprocess,sys
if importlib.util.find_spec("imageio_ffmpeg") is None:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","imageio-ffmpeg"])
import imageio_ffmpeg
plt.rcParams["animation.ffmpeg_path"]=imageio_ffmpeg.get_ffmpeg_exe()
EXPORT_MOVIES=True
if EXPORT_MOVIES:
    movie_path=simulate(4000,8100,start=10*N+5)
    fig=plt.figure(figsize=(13,4.5),layout="constrained")
    ax=fig.add_subplot(131,projection="3d",computed_zorder=False)
    ax.plot_surface(X,Y,energy,cmap="viridis",alpha=.65,zorder=1,linewidth=0)
    ball,=ax.plot([],[],[],"ro",ms=8,zorder=10);ax.set(xlabel="x",ylabel="y",zlabel="Potential",title="Particle on the energy surface")
    xy=fig.add_subplot(132);xy.contourf(X,Y,energy,25,cmap="viridis");basin_grid(xy)
    trail,=xy.plot([],[],"w-",lw=1);dot,=xy.plot([],[],"ro",ms=6);xy.set(xlabel="x",ylabel="y",aspect="equal")
    axh=fig.add_subplot(133)
    hist=axh.imshow(np.zeros((N,N)),origin="lower",extent=(-1,1,-1,1),cmap="magma",norm=PowerNorm(.3,vmin=0,vmax=1))
    axh.contour(X,Y,W,levels=5,colors="cyan",linewidths=.5)
    axh.set(title="Running occupancy",xlabel="x",ylabel="y");fig.colorbar(hist,ax=axh,label="Probability per grid cell")
    def update_mc(frame):
        t=frame*20;iy,ix=divmod(int(movie_path[t]),N)
        ball.set_data_3d([x[ix]],[y[iy]],[energy[iy,ix]])
        yy,xx=np.divmod(movie_path[max(0,t-100):t+1],N)
        trail.set_data(x[xx],y[yy]);dot.set_data([x[ix]],[y[iy]])
        hist.set_data(np.bincount(movie_path[:t+1],minlength=N*N).reshape(N,N)/(t+1))
        xy.set_title(f"Step {t}; 20 MC steps/frame")
    animation_mc=FuncAnimation(fig,update_mc,frames=201,interval=80)
    print("Rendering MC movie (this may take a few minutes)...", flush=True)
    animation_mc.save(str(OUTPUT/"Prinz_8well_MC.mp4"),writer=FFMpegWriter(fps=12),dpi=110);plt.close(fig)
    fig,axes=plt.subplots(2,2,figsize=(11,8),layout="constrained")
    im=axes[0,0].imshow(population[0].reshape(2,4),origin="lower",extent=(-1,1,-1,1),cmap="magma",norm=PowerNorm(.3,vmin=0,vmax=1))
    fig.colorbar(im,ax=axes[0,0],label="Basin probability")
    exc=axes[0,1].imshow((population[0]-pi).reshape(2,4),origin="lower",extent=(-1,1,-1,1),cmap="coolwarm",norm=SymLogNorm(linthresh=.001,vmin=-1,vmax=1))
    fig.colorbar(exc,ax=axes[0,1],label="Excess / depleted probability (symlog)")
    for ax in axes[0]:basin_grid(ax);ax.set(xlabel="x",ylabel="y")
    bars=axes[1,0].bar(np.arange(8),population[0],color="#0072b2")
    axes[1,0].plot(np.arange(8),pi,"k_",ms=16,label="Fitted equilibrium")
    axes[1,0].set(xticks=range(8),xticklabels=labels,ylim=(0,1),ylabel="Basin probability",title="Eight populations");axes[1,0].legend(fontsize=9)
    modebars=axes[1,1].bar(np.arange(2,9),a[1:],color="#cc8018")
    limit=1.3*max(abs(a[1:]));axes[1,1].set(ylim=(-limit,limit),xticks=range(2,9),xlabel="Mode",ylabel="Signed amplitude",title="All seven decaying amplitudes")
    axes[1,1].axhline(0,color="gray",lw=.7)
    annotations=[axes[1,1].text(i+2,0,"",ha="center",fontsize=8) for i in range(7)]
    def update_population(k):
        im.set_data(population[k].reshape(2,4));exc.set_data((population[k]-pi).reshape(2,4))
        axes[0,0].set_title(f"k={k}; {k*LAG:,} MC steps")
        axes[0,1].set_title(f"Imbalance; TV distance={TV[k]:.4f}")
        for bar,p in zip(bars,population[k]):bar.set_height(p)
        for i,(bar,text) in enumerate(zip(modebars,annotations),1):
            value=a[i]*vals[i]**k;bar.set_height(value)
            text.set_text(f"{value:.3f}");text.set_position((i+1,value+(.08 if value>=0 else -.15)))
    animation_population=FuncAnimation(fig,update_population,frames=range(N_LAGS+1),interval=80)
    print("MC movie ready. Rendering ensemble relaxation...", flush=True)
    animation_population.save(str(OUTPUT/"Prinz_8well_ensemble_relaxation.mp4"),writer=FFMpegWriter(fps=12),dpi=110);plt.close(fig)
    display(Video(str(OUTPUT/"Prinz_8well_MC.mp4"),embed=True))
    display(Video(str(OUTPUT/"Prinz_8well_ensemble_relaxation.mp4"),embed=True))

    print("Both movies are ready above.", flush=True)


## A compact homework
1. Run your own independent training and validation trajectories. Estimate the 8×8 matrix, show its stationary distribution, and check row sums and detailed balance.
2. Plot the four slowest nonstationary eigenmodes on the eight wells. Explain their exchange patterns without assigning physical meaning to an arbitrary overall sign.
3. Refit at several lags and plot implied timescales. Compare direct held-out transitions with MSM predictions; extend the eight return panels to selected off-diagonal entries or the full matrix.
4. Start all population in a different basin. Calculate every amplitude, reconstruct the initial population, and make a relaxation movie. Verify conservation and compare spectral reconstruction with repeated matrix multiplication.
5. Optional: subdivide the eight basins while keeping the fine-grid chain fixed. Increase sampling, check count connectivity, and compare errors against the exact reference. The MSM need not remain eight-state when addressing discretization bias.

**Interpretation:** this lesson keeps the displayed model small enough to inspect each entry. Simplicity does not guarantee short-lag Markov behavior. Longer trajectories reduce estimation error; refinement or longer lag can address information lost by coarse states. All propagation results are for the specified reversible MC model and its fitted MSM, not molecular time scales.